Day-08

Audit image/video files, labels/targets, duplicates and split integrity.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
dataset_path = "/content/drive/MyDrive/original_data"
print(os.listdir(dataset_path))

['images', 'annotations', 'README.txt']


In [ ]:
img_dir = os.path.join(dataset_path, "images")
anno_dir = os.path.join(dataset_path, "annotations")
print("images : ", len(os.listdir(img_dir)))
print("annotations : ", len(os.listdir(anno_dir)))

images :  7214
annotations :  7212


In [ ]:
print("SAMPLE IMAGE FILES:")
for f in sorted(os.listdir(img_dir))[:10]:
    print(f)

print("\nSAMPLE ANNOTATION FILES:")
for f in sorted(os.listdir(anno_dir))[:10]:
    print(f)

SAMPLE IMAGE FILES:
vid_000002_frame0000013.jpg
vid_000002_frame0000014.jpg
vid_000002_frame0000015.jpg
vid_000002_frame0000016.jpg
vid_000002_frame0000017.jpg
vid_000002_frame0000018.jpg
vid_000002_frame0000019.jpg
vid_000002_frame0000020.jpg
vid_000002_frame0000021.jpg
vid_000002_frame0000022.jpg

SAMPLE ANNOTATION FILES:
vid_000002_frame0000013.jpg.json
vid_000002_frame0000014.jpg.json
vid_000002_frame0000015.jpg.json
vid_000002_frame0000016.jpg.json
vid_000002_frame0000017.jpg.json
vid_000002_frame0000018.jpg.json
vid_000002_frame0000019.jpg.json
vid_000002_frame0000020.jpg.json
vid_000002_frame0000021.jpg.json
vid_000002_frame0000022.jpg.json


In [ ]:
print("IMAGE NAMES:")
for f in sorted(os.listdir(img_dir))[:10]:
    print(os.path.splitext(f)[0])

print("\nANNOTATION NAMES:")
for f in sorted(os.listdir(anno_dir))[:10]:
    print(os.path.splitext(f)[0])

IMAGE NAMES:
vid_000002_frame0000013
vid_000002_frame0000014
vid_000002_frame0000015
vid_000002_frame0000016
vid_000002_frame0000017
vid_000002_frame0000018
vid_000002_frame0000019
vid_000002_frame0000020
vid_000002_frame0000021
vid_000002_frame0000022

ANNOTATION NAMES:
vid_000002_frame0000013.jpg
vid_000002_frame0000014.jpg
vid_000002_frame0000015.jpg
vid_000002_frame0000016.jpg
vid_000002_frame0000017.jpg
vid_000002_frame0000018.jpg
vid_000002_frame0000019.jpg
vid_000002_frame0000020.jpg
vid_000002_frame0000021.jpg
vid_000002_frame0000022.jpg


In [ ]:
image_files = {
    os.path.splitext(f)[0]
    for f in os.listdir(img_dir)
    if f.lower().endswith(".jpg")
}
annotation_files = {
    f[:-9]
    for f in os.listdir(anno_dir)
    if f.lower().endswith(".jpg.json")
}

img_without_anno = image_files - annotation_files
anno_without_img = annotation_files - image_files

print("Total images:", len(image_files))
print("Total annotations:", len(annotation_files))
print("\nImages without annotations:", len(img_without_anno))
print("Annotations without images:", len(anno_without_img))

Total images: 7214
Total annotations: 7212

Images without annotations: 2
Annotations without images: 0


In [ ]:
print("images without annotations:")
for f in sorted(img_without_anno):
    print(f)

print("\nannotations without images:")
for f in sorted(anno_without_img):
    print(f)

images without annotations:
vid_000270_frame0000041 (1)
vid_000270_frame0000044 (1)

annotations without images:


In [ ]:
import json

json_files = [
    f for f in os.listdir(anno_dir)
    if f.lower().endswith(".jpg.json")
]
sample_json = os.path.join(anno_dir, json_files[0])
with open(sample_json, "r") as f:
    data = json.load(f)
print(json.dumps(data, indent=2)[:10000])

{
  "description": "",
  "tags": [],
  "size": {
    "height": 270,
    "width": 480
  },
  "objects": [
    {
      "id": 411140420,
      "classId": 1400966,
      "description": "",
      "geometryType": "bitmap",
      "labelerLogin": "jasonsf",
      "createdAt": "2020-04-06T08:17:01.709Z",
      "updatedAt": "2020-04-06T20:41:11.370Z",
      "tags": [
        {
          "id": 27641173,
          "name": "material",
          "value": "etc",
          "labelerLogin": "jasonsf",
          "createdAt": "2020-04-06T08:17:01.615Z",
          "updatedAt": "2020-04-06T08:17:01.615Z"
        },
        {
          "id": 27641175,
          "name": "crushed/broken",
          "value": null,
          "labelerLogin": "jasonsf",
          "createdAt": "2020-04-06T08:17:01.615Z",
          "updatedAt": "2020-04-06T08:17:01.615Z"
        },
        {
          "id": 27641177,
          "name": "decay",
          "value": null,
          "labelerLogin": "jasonsf",
          "createdAt": "2020

In [10]:
from PIL import Image
corrupt_images = []

for filename in os.listdir(img_dir):
    if filename.lower().endswith(".jpg"):
        path = os.path.join(img_dir, filename)

        try:
            with Image.open(path) as img:
                img.verify()
        except Exception:
            corrupt_images.append(filename)
print("corrupt images:", len(corrupt_images))

corrupt images: 0


In [11]:
from collections import Counter

resolution_counts = Counter()
for filename in os.listdir(img_dir):
    if filename.lower().endswith(".jpg"):
        path = os.path.join(img_dir, filename)
        try:
            with Image.open(path) as img:
                resolution_counts[img.size] += 1
        except:
            pass
print("different resolutions:",len(resolution_counts))
for resolution, count in resolution_counts.most_common(20):
    print(resolution, ":",count)

different resolutions: 2
(480, 270) : 3967
(480, 360) : 3247


In [13]:
invalid_json = []
valid_json = 0

for filename in os.listdir(anno_dir):
    if not filename.lower().endswith(".jpg.json"):
        continue
    path = os.path.join(anno_dir, filename)
    try:
        with open(path, "r") as f:
            json.load(f)
        valid_json += 1
    except Exception as e:
        invalid_json.append((filename, str(e)))

print("valid json files:", valid_json)
print("invalid json files:", len(invalid_json))
if invalid_json:
    print("\nExamples of invalid files:")
    for item in invalid_json[:10]:
        print(item)

Valid JSON files: 7212
Invalid JSON files: 0


In [14]:
empty_annotations = []
for filename in os.listdir(anno_dir):
    if not filename.lower().endswith(".jpg.json"):
        continue
    path = os.path.join(anno_dir, filename)
    try:
        with open(path, "r") as f:
            data = json.load(f)
        objects = data.get("objects", [])
        if len(objects) == 0:
            empty_annotations.append(filename)
    except Exception:
        pass
print("empty annotation files:", len(empty_annotations))
if empty_annotations:
    print("\nExamples:")
    for filename in empty_annotations[:20]:
        print(filename)

Empty annotation files: 0


In [16]:
class_counts = Counter()
for filename in os.listdir(anno_dir):
    if not filename.lower().endswith(".jpg.json"):
        continue
    path = os.path.join(anno_dir, filename)
    try:
        with open(path, "r") as f:
            data = json.load(f)
        for obj in data.get("objects", []):
            class_name = obj.get("classTitle", "UNKNOWN")
            class_counts[class_name] += 1
    except Exception:
        pass
print("Number of classes:", len(class_counts))
print("\nObjects per class:")
for class_name, count in class_counts.most_common():
    print(f"{class_name}: {count}")

Number of classes: 4

Objects per class:
trash: 5652
rov: 3447
bio: 2805
unknown: 576


In [12]:
import hashlib
from collections import defaultdict

hash_to_files = defaultdict(list)

for filename in os.listdir(img_dir):
    if not filename.lower().endswith(".jpg"):
        continue
    path = os.path.join(img_dir, filename)
    with open(path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    hash_to_files[file_hash].append(filename)
duplicate_groups = {
    h: files
    for h, files in hash_to_files.items()
    if len(files) > 1
}

duplicate_image_count = sum(len(files) - 1 for files in duplicate_groups.values())

print("Total images:", len(image_files))
print("Unique image contents:", len(hash_to_files))
print("Duplicate groups:", len(duplicate_groups))
print("Duplicate images:", duplicate_image_count)

Total images: 7214
Unique image contents: 7212
Duplicate groups: 2
Duplicate images: 2


In [17]:
import re
video_frames = Counter()
for filename in image_files:
    match = re.match(r"(vid_\d+)_frame(\d+)", filename)
    if match:
        video_id = match.group(1)
        video_frames[video_id] += 1
print("Number of source videos:", len(video_frames))
print("\nFrames per video:")
for video_id, count in sorted(video_frames.items()):
    print(video_id, ":", count)

Number of source videos: 312

Frames per video:
vid_000002 : 11
vid_000003 : 22
vid_000004 : 9
vid_000005 : 6
vid_000020 : 3
vid_000021 : 25
vid_000022 : 13
vid_000023 : 10
vid_000024 : 6
vid_000025 : 13
vid_000026 : 19
vid_000027 : 9
vid_000028 : 43
vid_000029 : 7
vid_000030 : 20
vid_000031 : 40
vid_000032 : 5
vid_000034 : 13
vid_000035 : 43
vid_000036 : 31
vid_000037 : 15
vid_000038 : 30
vid_000039 : 17
vid_000040 : 38
vid_000041 : 30
vid_000042 : 21
vid_000043 : 12
vid_000044 : 15
vid_000045 : 51
vid_000046 : 14
vid_000047 : 25
vid_000048 : 29
vid_000049 : 11
vid_000050 : 20
vid_000051 : 1
vid_000052 : 38
vid_000053 : 14
vid_000054 : 14
vid_000055 : 16
vid_000063 : 11
vid_000067 : 2
vid_000068 : 4
vid_000069 : 15
vid_000070 : 8
vid_000072 : 11
vid_000073 : 18
vid_000074 : 38
vid_000075 : 52
vid_000076 : 41
vid_000077 : 43
vid_000078 : 9
vid_000079 : 32
vid_000080 : 47
vid_000081 : 31
vid_000082 : 24
vid_000083 : 5
vid_000084 : 20
vid_000085 : 28
vid_000086 : 46
vid_000087 : 2
vid_00